In [ ]:
import geopandas as gpd

from benchmark_pipeline.path import DATA
from benchmark_pipeline.generator.template_question.multihop.generator_multihop import make_multihop_questions
from benchmark_pipeline.generator.template_question.multihop.make_multihop_text import make_text_multihop

In [2]:
repo = "benchmark_1K_after_correctionbug"
df_question = gpd.read_parquet(DATA / repo / "tables" / "df_question.parquet")
df_street = gpd.read_parquet(DATA / repo / "tables" / "df_street.parquet")
df_area = gpd.read_parquet(DATA / repo / "tables" / "df_area.parquet")
df_osm = gpd.read_parquet(DATA / repo / "tables" / "df_osm.parquet")

In [3]:
%load_ext autoreload
%autoreload 2
import numpy as np

id_start = df_question[df_question["values"].map(tuple) == ("station",)].index[5]


backward_index, df_question, df_multihop = make_multihop_questions(id_start, {5:0, 4: 3, 3: 3, 2: 3, 1: 2}, df_question, df_osm, df_street, df_area)

In [4]:
df_multihop = None
k=1000
for id_start in df_question.sample(k, random_state=0).index:
    backward_index, df_question, df_multihop = make_multihop_questions(
        id_start, {5: 0, 4: 3, 3: 3, 2: 3, 1: 2}, df_question, df_osm, df_street, df_area, df_multihop=df_multihop)
df_multihop = make_text_multihop(df_multihop, df_question)


In [5]:
from benchmark_pipeline.config import MULTIHOP_COLUMNS


def dicts_to_lists(df):
    # Parquet only takes str dict keys: node i is stored at position i - 1, each dic_query value as a list
    def as_list(value):
        return value if isinstance(value, list) else [] if value is None else [value]
    return df.assign(
        template=df["template"].map(lambda d: [d[node] for node in sorted(d)] if isinstance(d, dict) else None),
        dic_query=df["dic_query"].map(lambda d: [as_list(d[node]) for node in sorted(d)] if isinstance(d, dict) else None))


multihop_dir = DATA / "multihop"
multihop_dir.mkdir(parents=True, exist_ok=True)
# Lateral rows keep template columns such as area_geometry, which parquet rejects
dicts_to_lists(df_question[MULTIHOP_COLUMNS]).to_parquet(multihop_dir / "df_question.parquet")
dicts_to_lists(df_multihop).to_parquet(multihop_dir / "df_multihop.parquet")

In [15]:
def sample_by_depth(df_multihop, k, seed=0):
    """k random questions per depth, all of them when a depth has fewer.

    Args:
        df_multihop (DataFrame): multihop questions, `template` as a dict or a list.
        k (int): questions kept per depth.
        seed (int): random seed.

    Returns:
        DataFrame: `query_texte` and `answers`, sorted by depth.
    """
    # template has one entry per node, plus a closing 0
    depth = df_multihop["template"].map(len) - 1
    shuffled_depth = depth.sample(frac=1, random_state=seed)
    kept_depth = shuffled_depth.groupby(shuffled_depth).head(k).sort_values(kind="stable")
    dump = df_multihop.loc[kept_depth.index, ["description_textuelle", "answers"]]
    dump = dump.rename(columns={"description_textuelle": "query"}).assign(depth=kept_depth)
    return dump

In [ ]:
def keep_questions(question_index, max_questions, random_generator):
    """At most `max_questions` labels drawn at random from `question_index`."""
    if len(question_index) <= max_questions:
        return list(question_index)
    return list(random_generator.choice(question_index, size=max_questions, replace=False))


def make_multihop_questions(
        id_question, 
        question_template, 
        df_question, 
        df_osm, 
        df_all, 
        max_questions_per_node=3,
        seed=0,
    ):
    """Build multihop questions following `question_template`, with `add_backward_hop` and `add_lateral_hop`.

    Args:
        max_questions_per_node (int): questions of a node the chain continues from; every question
            created by the hops stays in df_multihop.

    Returns:
        tuple: labels of the last node, `df_question`, `df_multihop`.
    """
    random_generator = np.random.default_rng(seed)
    df_multihop = pd.DataFrame()
    nodes = sorted(question_template, reverse=True)
    backward_index = [id_question]
    for node in nodes[1:]:
        print(node)
        nb_hop = question_template[node]
        if nb_hop >= 2:
            lateral_index = []
            for index in backward_index:
                new_index, df_question, df_multihop = add_lateral_hop(index, nb_hop, df_question, df_all, df_multihop)
                lateral_index += new_index
            backward_index = keep_questions(lateral_index, max_questions_per_node, random_generator)
        next_index = []
        if node > 1:
            for index in backward_index:
                new_index, df_multihop = add_backward_hop(index, df_osm, df_question, df_multihop)
                next_index += new_index
            backward_index = keep_questions(next_index, max_questions_per_node, random_generator)
    
    return backward_index, df_question, df_multihop

In [ ]:
import numpy as np

id_start = df_question_light[df_question_light["values"]==('station',)].index[5]

backward_index, df_question, df_multihop = make_multihop_questions(id_start, {5:0, 4: 1, 3: 2, 2: 1, 1: 2}, df_question_light, df_osm, df_all)

In [ ]:
from benchmark_pipeline.generator.template_question.composite.generator_qcomposite import question_en_texte


def attributes_text(features, values):
    """« tapas food with outdoor seating »: the attributes of a question, as in its text."""
    return question_en_texte("pois", tuple(features), tuple(values), FEATURES)


def make_text_multihop(df_multihop):
    """Add `description_textuelle` to df_multihop: each question as short sentences, one per node,
    from the final question to the starting one.

    The starting questions are read in the global `df_question`, the one returned by make_multihop_questions.
    """
    multihop = df_multihop.set_index("index")
    nodes_cache = {}

    def text_of(id_query):
        if id_query in multihop.index:
            return multihop.loc[id_query, "query_texte"]
        return df_question.loc[id_query, "query_texte"]

    def nodes_of(id_query):
        """[{sentence, short}] from the node of `id_query` down to the starting question."""
        if id_query in nodes_cache:
            return nodes_cache[id_query]
        if id_query not in multihop.index:
            question = df_question.loc[id_query]
            nodes = [{"sentence": question["query_texte"],
                      "short": attributes_text(question["features"][:1], question["values"][:1])}]
        else:
            row = multihop.loc[id_query]
            below = nodes_of(row["index_origin"])
            origin_text = text_of(row["index_origin"])
            if isinstance(row.get("index_lateral"), list):
                relations = row["query_texte"][len("[" + origin_text + " and "):-1]
                nodes = [{**below[0], "sentence": below[0]["sentence"] + " and " + relations}] + below[1:]
            else:
                head = attributes_text(row["features"], row["values"])
                relation = row["query_texte"][len(head) + 1:-len(origin_text) - 1]
                nodes = [{"sentence": f"{head} {relation} {below[0]['short']}",
                          "short": attributes_text(row["features"][:1], row["values"][:1])}] + below
        nodes_cache[id_query] = nodes
        return nodes

    def description(id_query):
        sentences = [" ".join(node["sentence"].split()) for node in nodes_of(id_query)]
        return " ".join(sentence[0].upper() + sentence[1:] + "." for sentence in sentences)

    df_multihop = df_multihop.copy()
    df_multihop["description_textuelle"] = [description(id_query) for id_query in df_multihop["index"]]
    return df_multihop


In [ ]:
df_multihop = make_text_multihop(df_multihop)